# Early-Layer-SAE-Feature-Forecasting: Step 1 - smoke test

Verifies the full activation-extraction pipeline:

1. Load Gemma-2-2B (base) via `transformer_lens`
2. Load a GemmaScope residual-stream SAE
3. Forward pass with cached residual streams at chosen layers
4. Encode the late-layer residual through the SAE and inspect top-activating features per token

**Runtime:** Change runtime type to **T4 GPU**.

**Auth:** Gemma-2-2B is gated. Before running:
1. Accept the license at https://huggingface.co/google/gemma-2-2b
2. Generate a read token at https://huggingface.co/settings/tokens
3. Paste the token in the login cell below

Step 1 is complete when the final cell prints non-zero SAE feature activations per token.

## 1. Install dependencies

Takes ~1-2 min on a fresh Colab runtime.

In [ ]:
!pip install -q sae_lens accelerate

## 2. Authenticate with Hugging Face

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Load Gemma-2-2B (via `transformers`, not `transformer_lens`)

Colab free tier has only ~12.7 GB **host RAM** (not VRAM). `transformer_lens` loads weights to CPU first and then reformats them into its own naming convention; that briefly holds two copies, which OOMs the kernel even with `from_pretrained_no_processing` and bf16.

Plain `transformers` with `device_map='cuda'` streams weights layer-by-layer directly to GPU, so CPU RAM never holds the full model. We use manual `register_forward_hook` calls on `model.model.layers[i]` to grab the post-block residual stream, which is the activation site GemmaScope residual SAEs were trained on (same as TransformerLens's `hook_resid_post`).

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: no GPU detected. Switch runtime to T4 GPU before continuing.')

MODEL_NAME = 'google/gemma-2-2b'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=device,            # streams weights directly to GPU
    low_cpu_mem_usage=True,       # keeps CPU footprint small during load
)
model.eval()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU mem allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

n_layers = model.config.num_hidden_layers
d_model = model.config.hidden_size
print(f'Loaded. n_layers={n_layers}, d_model={d_model}')

## 4. Load a GemmaScope SAE (layer 20, residual stream, width 16k)

If the `release` / `sae_id` strings below don't match the registry, the next cell prints what's available so you can adjust.

In [ ]:
import torch
from sae_lens import SAE

# Defensive: define device if cell 6 wasn't run / kernel was restarted
try:
    device
except NameError:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'(device was not set; defaulting to {device})')

RELEASE = 'gemma-scope-2b-pt-res-canonical'
SAE_ID = 'layer_20/width_16k/canonical'

try:
    sae, cfg_dict, sparsity = SAE.from_pretrained(
        release=RELEASE,
        sae_id=SAE_ID,
        device=device,
    )
    print(f'SAE loaded. d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}, dtype={sae.W_enc.dtype}')
    print(f"hook_point={cfg_dict.get('hook_point') or cfg_dict.get('hook_name')}")
    print(f"hook_layer={cfg_dict.get('hook_layer')}")
except Exception as e:
    print(f'Load failed: {type(e).__name__}: {e}')
    print()
    print('Known GemmaScope releases for Gemma-2-2B:')
    for r in [
        'gemma-scope-2b-pt-res-canonical',   # residual stream, best L0 per layer
        'gemma-scope-2b-pt-res',             # residual stream, multiple L0 variants
        'gemma-scope-2b-pt-mlp-canonical',
        'gemma-scope-2b-pt-mlp',
        'gemma-scope-2b-pt-att-canonical',
        'gemma-scope-2b-pt-att',
    ]:
        print(f'  {r}')
    print()
    print('SAE IDs follow pattern: layer_<L>/width_<W>/canonical')
    print('  (or layer_<L>/width_<W>/average_l0_<X> for non-canonical)')
    print()
    # Try dynamic discovery across sae_lens versions (the path moved between releases)
    import importlib
    directory = None
    for path in [
        'sae_lens.get_pretrained_saes_directory',
        'sae_lens.toolkit.pretrained_saes_directory.get_pretrained_saes_directory',
        'sae_lens.pretrained_saes_directory.get_pretrained_saes_directory',
    ]:
        try:
            module_path, attr = path.rsplit('.', 1)
            mod = importlib.import_module(module_path)
            directory = getattr(mod, attr)()
            print(f'(Found registry via {path})')
            break
        except (ImportError, AttributeError):
            continue
    if directory is not None:
        print('Registry entries matching gemma-scope-2b:')
        for name in sorted(directory.keys()):
            if 'gemma-scope' in name.lower() and '2b' in name.lower():
                saes = list(directory[name].saes_map.keys())
                print(f'  {name}: {len(saes)} SAEs (sample: {saes[:3]})')
    raise

## 5. Forward pass with manual residual-stream hooks

We register forward hooks on the two transformer blocks we care about (layer 5 + layer 20). In HF's Gemma2 implementation, `model.model.layers[i].forward()` returns a tuple whose first element is the post-block residual stream, i.e. exactly the activation site GemmaScope residual SAEs expect.

In [ ]:
prompt = "The customer's complaint, while rude, raises a valid point."
inputs = tokenizer(prompt, return_tensors='pt').to(device)
token_strs = [tokenizer.decode([t]) for t in inputs.input_ids[0]]
print(f'Token count: {inputs.input_ids.shape[1]}')
print(f'Tokens: {token_strs}')

# Capture post-block residual stream at layers 5 and 20
cache = {}
def make_hook(name):
    def hook(module, input, output):
        # Gemma2DecoderLayer.forward returns (hidden_states, ...); hidden_states
        # is the residual stream after both attention and MLP have been added.
        hidden = output[0] if isinstance(output, tuple) else output
        cache[name] = hidden.detach()
    return hook

hooks = []
hooks.append(model.model.layers[5].register_forward_hook(make_hook('layer_5_resid_post')))
hooks.append(model.model.layers[20].register_forward_hook(make_hook('layer_20_resid_post')))

try:
    with torch.no_grad():
        _ = model(**inputs)
finally:
    for h in hooks:
        h.remove()

print(f'cache keys: {list(cache.keys())}')
print(f'layer 5 resid shape:  {tuple(cache["layer_5_resid_post"].shape)}')
print(f'layer 20 resid shape: {tuple(cache["layer_20_resid_post"].shape)}')

## 6. Encode the layer-20 residual through the SAE

Smoke-test success criterion: for each token, a small fraction of the ~16k features are active (typical L0 for GemmaScope canonical SAEs is ~50-200 per token), and the top features have sane-looking magnitudes.

In [ ]:
resid_20 = cache['layer_20_resid_post'].to(sae.W_enc.dtype)

with torch.no_grad():
    feature_acts = sae.encode(resid_20)

print(f'feature_acts shape: {tuple(feature_acts.shape)}  # (batch, seq, d_sae)')
print()

for tok_idx in range(feature_acts.shape[1]):
    acts = feature_acts[0, tok_idx].float()
    nonzero = int((acts > 0).sum().item())
    top = torch.topk(acts, k=5)
    top_pairs = [(int(i), round(float(v), 3)) for i, v in zip(top.indices, top.values)]
    print(f'{tok_idx:>3}  {repr(token_strs[tok_idx]):<14}  active={nonzero:>4}  top-5: {top_pairs}')

## Done

If the cell above printed feature activations for every token (with ~tens to hundreds of non-zero features each), the pipeline works end-to-end.

**Next step (Step 2):** browse Neuronpedia at https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k to identify safety-flavored target features. You'll bring back 3-5 feature indices for use in Step 3 (full activation cache extraction).